
# HAT Max-Performance: Long Same-Task Pretraining on PanCollection WV3

## الهدف

تقوية أفضل HAT-PAN Fusion قبل نقلها إلى البيانات المحلية ذات الست حزم.

بدل التدريب السابق القصير، هذه الـNotebook تستخدم:

```text
PanCollection WorldView-3
Train ≈ 9,714 samples
Validation ≈ 1,080 samples
8 MS bands + PAN
Scale ×4
```

وتضيف:

```text
Long continuation training
Full float32 stability
Charbonnier reconstruction loss
Small SSIM loss
Small spectral-angle loss
Low-resolution consistency loss
EMA weights
Warmup + cosine learning-rate schedule
Gradient clipping
Automatic resume
Best-by-PSNR and best-balanced checkpoints
```

## لماذا لا نستخدم GAN هنا؟

هذه المرحلة هدفها تعظيم:

```text
PSNR
SSIM
SAM
ERGAS
```

الـGAN قد يحسن الشكل البصري، لكنه غالبًا يضحي بجزء من PSNR أو الدقة الطيفية. لذلك HAT تظل Distortion-Optimized، وتجربة GAN تكون منفصلة على Sentinel-2.


In [1]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("فعّل T4 GPU من Runtime → Change runtime type.")

device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:

from pathlib import Path
import shutil
import subprocess
import importlib.util
import sys

%cd /content

HAT_REPO = Path("/content/HAT")

if HAT_REPO.exists():
    shutil.rmtree(HAT_REPO)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/XPixelGroup/HAT.git",
        str(HAT_REPO),
    ],
    check=True,
)

!pip install -q einops timm h5py pandas matplotlib tqdm huggingface_hub pytorch-msssim

original_arch = HAT_REPO / "hat" / "archs" / "hat_arch.py"
standalone_arch = Path("/content/hat_arch_standalone.py")

source = original_arch.read_text(encoding="utf-8")

source = source.replace(
    "from basicsr.utils.registry import ARCH_REGISTRY",
    """class _SimpleRegistry:
    def register(self):
        def decorator(obj):
            return obj
        return decorator
ARCH_REGISTRY = _SimpleRegistry()"""
)

source = source.replace(
    "from basicsr.archs.arch_util import to_2tuple, trunc_normal_",
    "from timm.layers import to_2tuple, trunc_normal_"
)

standalone_arch.write_text(source, encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "hat_arch_standalone",
    standalone_arch,
)

hat_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hat_module)

HAT = hat_module.HAT

print("Official HAT imported.")


/content
Official HAT imported.


In [3]:

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Super_Resolution_28-07-2026"
)

WV3_DIR = (
    PROJECT_DIR
    / "Public_Datasets"
    / "PanCollection_WV3"
)

OLD_PRETRAIN_DIR = (
    PROJECT_DIR
    / "WV3_HAT_Pretraining"
)

BASE_CHECKPOINT = (
    OLD_PRETRAIN_DIR
    / "best_wv3_hat_pan_pretrained.pth"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "WV3_HAT_Long_Pretraining"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PSNR_PATH = (
    OUTPUT_DIR
    / "best_psnr_wv3_hat_long.pth"
)

BEST_BALANCED_PATH = (
    OUTPUT_DIR
    / "best_balanced_wv3_hat_long.pth"
)

LAST_PATH = (
    OUTPUT_DIR
    / "last_wv3_hat_long.pth"
)

HISTORY_PATH = (
    OUTPUT_DIR
    / "long_pretraining_history.json"
)

FINAL_METRICS_PATH = (
    OUTPUT_DIR
    / "final_validation_metrics.json"
)

print("WV3:", WV3_DIR)
print("Base checkpoint:", BASE_CHECKPOINT)
print("Output:", OUTPUT_DIR)

if not BASE_CHECKPOINT.exists():
    raise FileNotFoundError(
        "شغّل Notebook 07 أولًا أو ضع أفضل WV3 HAT checkpoint في المسار المطبوع."
    )


Mounted at /content/drive
WV3: /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/PanCollection_WV3
Base checkpoint: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_HAT_Pretraining/best_wv3_hat_pan_pretrained.pth
Output: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_HAT_Long_Pretraining


## تنزيل WV3 فقط لو الملفات غير موجودة

In [4]:

import h5py

h5_files = sorted(WV3_DIR.rglob("*.h5")) if WV3_DIR.exists() else []

def locate_files(files):
    train = [
        path for path in files
        if (
            "wv3" in path.name.lower()
            and "train" in path.name.lower()
            and "valid" not in path.name.lower()
            and "test" not in path.name.lower()
        )
    ]

    valid = [
        path for path in files
        if (
            "wv3" in path.name.lower()
            and (
                "valid" in path.name.lower()
                or "validation" in path.name.lower()
            )
        )
    ]

    return train, valid

train_candidates, valid_candidates = locate_files(h5_files)

if not train_candidates or not valid_candidates:
    from huggingface_hub import HfApi, snapshot_download

    repo_id = "elsting/PanCollection"
    api = HfApi()

    repo_files = api.list_repo_files(
        repo_id=repo_id,
        repo_type="dataset",
    )

    selected = [
        name
        for name in repo_files
        if (
            "wv3" in name.lower()
            and "training_data/" in name.lower()
            and name.lower().endswith(".h5")
        )
    ]

    if not selected:
        raise RuntimeError("لم يتم العثور على WV3 H5 في PanCollection.")

    WV3_DIR.mkdir(parents=True, exist_ok=True)

    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        allow_patterns=selected,
        local_dir=str(WV3_DIR),
    )

    h5_files = sorted(WV3_DIR.rglob("*.h5"))
    train_candidates, valid_candidates = locate_files(h5_files)

TRAIN_H5 = train_candidates[0]
VALID_H5 = valid_candidates[0]

for path in [TRAIN_H5, VALID_H5]:
    print("\n", path)

    with h5py.File(path, "r") as file:
        for key in ["ms", "pan", "gt", "lms"]:
            if key in file:
                print(key, file[key].shape, file[key].dtype)



 /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/PanCollection_WV3/training_data/train_wv3_9714.h5
ms (9714, 8, 16, 16) float64
pan (9714, 1, 64, 64) float64
gt (9714, 8, 64, 64) float64
lms (9714, 8, 64, 64) float64

 /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/PanCollection_WV3/training_data/valid_wv3_9714.h5
ms (1080, 8, 16, 16) float64
pan (1080, 1, 64, 64) float64
gt (1080, 8, 64, 64) float64
lms (1080, 8, 64, 64) float64


## إعداد التدريب

In [24]:

import os
import json
import math
import random
import numpy as np

RUN_PROFILE = "smoke"
# "smoke" للتأكد من سلامة التنفيذ
# "long" للتدريب القوي
# "max" لأطول تشغيل عملي مع Resume


SEED = 42
MAX_DN = 2047.0

BATCH_SIZE = 2
ACCUMULATION_STEPS = 4
NUM_WORKERS = 0

if RUN_PROFILE == "smoke":
    TARGET_TOTAL_EPOCHS = 1
    VALIDATE_EVERY = 1

elif RUN_PROFILE == "long":
    TARGET_TOTAL_EPOCHS = 150
    VALIDATE_EVERY = 2

elif RUN_PROFILE == "max":
    TARGET_TOTAL_EPOCHS = 300
    VALIDATE_EVERY = 3

else:
    raise ValueError(RUN_PROFILE)

BASE_LR = 5e-5
MIN_LR = 1e-6
WARMUP_EPOCHS = 5
WEIGHT_DECAY = 1e-6

CHARBONNIER_EPS = 1e-3
SSIM_WEIGHT = 0.04
SAM_WEIGHT = 0.015
LR_CONSISTENCY_WEIGHT = 0.05

EMA_DECAY = 0.999
GRADIENT_CLIP = 1.0

# Previous experiments showed HAT could become NaN under FP16.
USE_AMP = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Profile:", RUN_PROFILE)
print("Target total epochs:", TARGET_TOTAL_EPOCHS)
print("Effective batch:", BATCH_SIZE * ACCUMULATION_STEPS)
print("Float32 training:", not USE_AMP)


Profile: smoke
Target total epochs: 1
Effective batch: 8
Float32 training: True


In [25]:

from torch.utils.data import Dataset, DataLoader


class WV3Dataset(Dataset):
    def __init__(self, h5_path, augment=False):
        self.h5_path = Path(h5_path)
        self.augment = augment
        self._file = None

        with h5py.File(self.h5_path, "r") as file:
            for key in ["ms", "pan", "gt"]:
                if key not in file:
                    raise KeyError(f"{key} missing from {self.h5_path}")

            self.length = file["ms"].shape[0]

            print(
                self.h5_path.name,
                "samples:",
                self.length,
                "| MS:",
                file["ms"].shape,
                "| PAN:",
                file["pan"].shape,
                "| GT:",
                file["gt"].shape,
            )

    def _open(self):
        if self._file is None:
            self._file = h5py.File(self.h5_path, "r")

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        self._open()

        ms = np.asarray(
            self._file["ms"][index],
            dtype=np.float32,
        )

        pan = np.asarray(
            self._file["pan"][index],
            dtype=np.float32,
        )

        gt = np.asarray(
            self._file["gt"][index],
            dtype=np.float32,
        )

        ms = torch.from_numpy(
            np.clip(ms / MAX_DN, 0, 1)
        ).float()

        pan = torch.from_numpy(
            np.clip(pan / MAX_DN, 0, 1)
        ).float()

        gt = torch.from_numpy(
            np.clip(gt / MAX_DN, 0, 1)
        ).float()

        if self.augment:
            if random.random() < 0.5:
                ms = torch.flip(ms, dims=[2])
                pan = torch.flip(pan, dims=[2])
                gt = torch.flip(gt, dims=[2])

            if random.random() < 0.5:
                ms = torch.flip(ms, dims=[1])
                pan = torch.flip(pan, dims=[1])
                gt = torch.flip(gt, dims=[1])

            rotations = random.randint(0, 3)

            if rotations:
                ms = torch.rot90(ms, rotations, dims=[1, 2])
                pan = torch.rot90(pan, rotations, dims=[1, 2])
                gt = torch.rot90(gt, rotations, dims=[1, 2])

        return {
            "lr_ms": ms,
            "pan": pan,
            "target": gt,
        }


train_dataset = WV3Dataset(TRAIN_H5, augment=True)
val_dataset = WV3Dataset(VALID_H5, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


train_wv3_9714.h5 samples: 9714 | MS: (9714, 8, 16, 16) | PAN: (9714, 1, 64, 64) | GT: (9714, 8, 64, 64)
valid_wv3_9714.h5 samples: 1080 | MS: (1080, 8, 16, 16) | PAN: (1080, 1, 64, 64) | GT: (1080, 8, 64, 64)
Train batches: 4857
Validation batches: 540


## نفس أفضل معمارية HAT-PAN Fusion

In [26]:

import torch.nn as nn
import torch.nn.functional as F


class RefinementBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, 1, 1),
        )

    def forward(self, tensor):
        return tensor + self.block(tensor)


class WV3HATPanFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.hat = HAT(
            upscale=4,
            in_chans=9,
            img_size=16,
            window_size=16,
            compress_ratio=3,
            squeeze_factor=30,
            conv_scale=0.01,
            overlap_ratio=0.5,
            img_range=1.0,
            depths=[6, 6, 6, 6, 6, 6],
            embed_dim=180,
            num_heads=[6, 6, 6, 6, 6, 6],
            mlp_ratio=2,
            upsampler="pixelshuffle",
            resi_connection="1conv",
            use_checkpoint=False,
            drop_path_rate=0.0,
        )

        self.hat.conv_last = nn.Conv2d(
            64,
            8,
            3,
            1,
            1,
        )

        self.refine_head = nn.Conv2d(
            17,
            64,
            3,
            1,
            1,
        )

        self.refine_body = nn.Sequential(
            RefinementBlock(64),
            RefinementBlock(64),
            RefinementBlock(64),
            RefinementBlock(64),
        )

        self.refine_tail = nn.Conv2d(
            64,
            8,
            3,
            1,
            1,
        )

    def forward(self, lr_ms, pan_hr):
        bicubic = F.interpolate(
            lr_ms,
            size=pan_hr.shape[-2:],
            mode="bicubic",
            align_corners=False,
        )

        pan_lr = F.interpolate(
            pan_hr,
            size=lr_ms.shape[-2:],
            mode="area",
        )

        residual = self.hat(
            torch.cat(
                [lr_ms, pan_lr],
                dim=1,
            )
        )

        coarse = bicubic + residual

        refinement = self.refine_tail(
            self.refine_body(
                self.refine_head(
                    torch.cat(
                        [
                            coarse,
                            bicubic,
                            pan_hr,
                        ],
                        dim=1,
                    )
                )
            )
        )

        return (coarse + refinement).clamp(0, 1)


model = WV3HATPanFusion().to(device)

print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)


Parameters: 21,095,008


## EMA

In [27]:

from copy import deepcopy


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema_model = deepcopy(model).eval()

        for parameter in self.ema_model.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        model_state = model.state_dict()
        ema_state = self.ema_model.state_dict()

        for key, ema_value in ema_state.items():
            source_value = model_state[key].detach()

            if ema_value.dtype.is_floating_point:
                ema_value.mul_(self.decay).add_(
                    source_value,
                    alpha=1.0 - self.decay,
                )
            else:
                ema_value.copy_(source_value)

    def state_dict(self):
        return self.ema_model.state_dict()

    def load_state_dict(self, state):
        self.ema_model.load_state_dict(state, strict=True)


ema = ModelEMA(model, decay=EMA_DECAY)


## Loss والمقاييس

In [28]:

from pytorch_msssim import ssim


def charbonnier_loss(prediction, target):
    difference = prediction - target

    return torch.sqrt(
        difference * difference
        + CHARBONNIER_EPS * CHARBONNIER_EPS
    ).mean()


def spectral_cosine_loss(prediction, target):
    dot = torch.sum(prediction * target, dim=1)

    denominator = torch.clamp(
        torch.linalg.vector_norm(prediction, dim=1)
        * torch.linalg.vector_norm(target, dim=1),
        min=1e-8,
    )

    cosine = torch.clamp(
        dot / denominator,
        -1,
        1,
    )

    return (1.0 - cosine).mean()


def training_loss(prediction, target, lr_ms):
    reconstruction = charbonnier_loss(
        prediction,
        target,
    )

    structural = 1.0 - ssim(
        prediction,
        target,
        data_range=1.0,
        size_average=True,
    )

    spectral = spectral_cosine_loss(
        prediction,
        target,
    )

    downsampled = F.interpolate(
        prediction,
        size=lr_ms.shape[-2:],
        mode="area",
    )

    lr_consistency = F.l1_loss(
        downsampled,
        lr_ms,
    )

    total = (
        reconstruction
        + SSIM_WEIGHT * structural
        + SAM_WEIGHT * spectral
        + LR_CONSISTENCY_WEIGHT * lr_consistency
    )

    components = {
        "charbonnier": reconstruction.detach(),
        "ssim_loss": structural.detach(),
        "spectral_loss": spectral.detach(),
        "lr_consistency": lr_consistency.detach(),
    }

    return total, components


def psnr_values(prediction, target):
    mse = torch.mean(
        (prediction - target) ** 2,
        dim=(1, 2, 3),
    )

    return 10.0 * torch.log10(
        1.0 / torch.clamp(mse, min=1e-12)
    )


def sam_values(prediction, target):
    dot = torch.sum(prediction * target, dim=1)

    denominator = torch.clamp(
        torch.linalg.vector_norm(prediction, dim=1)
        * torch.linalg.vector_norm(target, dim=1),
        min=1e-8,
    )

    cosine = torch.clamp(
        dot / denominator,
        -1 + 1e-7,
        1 - 1e-7,
    )

    return (
        torch.acos(cosine)
        * 180.0
        / math.pi
    ).mean(dim=(1, 2))


def ergas_values(prediction, target, scale=4):
    rmse = torch.sqrt(
        torch.mean(
            (prediction - target) ** 2,
            dim=(2, 3),
        )
    )

    target_mean = torch.mean(
        target,
        dim=(2, 3),
    ).abs().clamp_min(1e-6)

    return (
        100.0
        / scale
        * torch.sqrt(
            torch.mean(
                (rmse / target_mean) ** 2,
                dim=1,
            )
        )
    )


## Optimizer وWarmup + Cosine

In [29]:

from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR


def learning_rate_multiplier(epoch_index):
    epoch_number = epoch_index + 1

    if epoch_number <= WARMUP_EPOCHS:
        return epoch_number / max(WARMUP_EPOCHS, 1)

    progress = (
        epoch_number - WARMUP_EPOCHS
    ) / max(
        TARGET_TOTAL_EPOCHS - WARMUP_EPOCHS,
        1,
    )

    cosine = 0.5 * (
        1.0 + math.cos(math.pi * min(progress, 1.0))
    )

    minimum_ratio = MIN_LR / BASE_LR

    return (
        minimum_ratio
        + (1.0 - minimum_ratio) * cosine
    )


optimizer = AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=learning_rate_multiplier,
)


## Resume من التدريب الطويل أو بدءًا من أفضل WV3 القديم

In [30]:

start_epoch = 1
best_psnr = -float("inf")
best_balanced_score = -float("inf")
history = []

if LAST_PATH.exists():
    checkpoint = torch.load(
        LAST_PATH,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    if "ema_state_dict" in checkpoint:
        ema.load_state_dict(
            checkpoint["ema_state_dict"]
        )
    else:
        ema = ModelEMA(model, decay=EMA_DECAY)

    start_epoch = int(checkpoint["epoch"]) + 1
    best_psnr = float(
        checkpoint.get("best_psnr", best_psnr)
    )
    best_balanced_score = float(
        checkpoint.get(
            "best_balanced_score",
            best_balanced_score,
        )
    )
    history = checkpoint.get("history", [])

    print("Resuming long run from epoch:", start_epoch)

else:
    base_checkpoint = torch.load(
        BASE_CHECKPOINT,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        base_checkpoint["model_state_dict"],
        strict=True,
    )

    ema = ModelEMA(model, decay=EMA_DECAY)

    old_epoch = int(
        base_checkpoint.get("epoch", 0)
    )

    print(
        "Loaded previous WV3 checkpoint. "
        "Recorded old epoch:",
        old_epoch,
    )

print("Training:", start_epoch, "→", TARGET_TOTAL_EPOCHS)


Loaded previous WV3 checkpoint. Recorded old epoch: 5
Training: 1 → 1


In [31]:

from tqdm.auto import tqdm


def save_checkpoint(path, epoch, validation):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "ema_state_dict": ema.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch": epoch,
            "best_psnr": best_psnr,
            "best_balanced_score": best_balanced_score,
            "validation": validation,
            "history": history,
            "architecture": "WV3 HAT-PAN Fusion 8-band",
            "loss_weights": {
                "ssim": SSIM_WEIGHT,
                "sam": SAM_WEIGHT,
                "lr_consistency": LR_CONSISTENCY_WEIGHT,
            },
        },
        path,
    )


@torch.inference_mode()
def evaluate(evaluation_model):
    evaluation_model.eval()

    results = {
        "l1": [],
        "psnr": [],
        "ssim": [],
        "sam": [],
        "ergas": [],
    }

    for batch in tqdm(
        val_loader,
        desc="WV3 validation",
        leave=False,
    ):
        lr_ms = batch["lr_ms"].to(device)
        pan = batch["pan"].to(device)
        target = batch["target"].to(device)

        prediction = evaluation_model(
            lr_ms,
            pan,
        ).clamp(0, 1)

        results["l1"].append(
            F.l1_loss(prediction, target).item()
        )

        results["psnr"].extend(
            psnr_values(prediction, target).cpu().tolist()
        )

        results["sam"].extend(
            sam_values(prediction, target).cpu().tolist()
        )

        results["ergas"].extend(
            ergas_values(prediction, target).cpu().tolist()
        )

        batch_ssim = ssim(
            prediction,
            target,
            data_range=1.0,
            size_average=False,
        )

        if batch_ssim.ndim == 0:
            results["ssim"].append(
                float(batch_ssim.item())
            )
        else:
            results["ssim"].extend(
                batch_ssim.cpu().tolist()
            )

    return {
        key: float(np.mean(values))
        for key, values in results.items()
    }


In [32]:

for epoch in range(start_epoch, TARGET_TOTAL_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    epoch_losses = []
    component_totals = {
        "charbonnier": [],
        "ssim_loss": [],
        "spectral_loss": [],
        "lr_consistency": [],
    }

    progress = tqdm(
        train_loader,
        desc=f"WV3 HAT {epoch}/{TARGET_TOTAL_EPOCHS}",
        leave=False,
    )

    for batch_index, batch in enumerate(progress, start=1):
        lr_ms = batch["lr_ms"].to(
            device,
            non_blocking=True,
        )

        pan = batch["pan"].to(
            device,
            non_blocking=True,
        )

        target = batch["target"].to(
            device,
            non_blocking=True,
        )

        prediction = model(lr_ms, pan)

        if not torch.isfinite(prediction).all():
            raise FloatingPointError(
                f"Non-finite prediction at epoch {epoch}, batch {batch_index}"
            )

        loss, components = training_loss(
            prediction,
            target,
            lr_ms,
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss at epoch {epoch}, batch {batch_index}"
            )

        (loss / ACCUMULATION_STEPS).backward()

        should_step = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(train_loader)
        )

        if should_step:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRADIENT_CLIP,
            )

            optimizer.step()
            ema.update(model)
            optimizer.zero_grad(set_to_none=True)

        epoch_losses.append(loss.item())

        for key, value in components.items():
            component_totals[key].append(
                float(value.item())
            )

        progress.set_postfix(
            loss=f"{loss.item():.5f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
        )

    scheduler.step()

    validation = None

    if (
        epoch == start_epoch
        or epoch % VALIDATE_EVERY == 0
        or epoch == TARGET_TOTAL_EPOCHS
    ):
        validation = evaluate(
            ema.ema_model
        )

    record = {
        "epoch": epoch,
        "train_loss": float(np.mean(epoch_losses)),
        "learning_rate": optimizer.param_groups[0]["lr"],
        "components": {
            key: float(np.mean(values))
            for key, values in component_totals.items()
        },
        "validation": validation,
    }

    history.append(record)

    print(
        f"Epoch {epoch:03d} | "
        f"Train {record['train_loss']:.6f}",
        end="",
    )

    if validation is not None:
        balanced_score = (
            validation["psnr"]
            + 2.0 * validation["ssim"]
            - 0.08 * validation["sam"]
            - 0.05 * validation["ergas"]
        )

        print(
            f" | PSNR {validation['psnr']:.3f}"
            f" | SSIM {validation['ssim']:.5f}"
            f" | SAM {validation['sam']:.3f}°"
            f" | ERGAS {validation['ergas']:.3f}"
            f" | L1 {validation['l1']:.5f}"
        )

        if validation["psnr"] > best_psnr:
            best_psnr = validation["psnr"]

            save_checkpoint(
                BEST_PSNR_PATH,
                epoch,
                validation,
            )

            print("Saved best PSNR checkpoint.")

        if balanced_score > best_balanced_score:
            best_balanced_score = balanced_score

            save_checkpoint(
                BEST_BALANCED_PATH,
                epoch,
                validation,
            )

            print("Saved best balanced checkpoint.")

    else:
        print()

    save_checkpoint(
        LAST_PATH,
        epoch,
        validation,
    )

    with open(HISTORY_PATH, "w", encoding="utf-8") as file:
        json.dump(
            history,
            file,
            indent=2,
            ensure_ascii=False,
        )

print("Long training completed.")


WV3 HAT 1/1:   0%|          | 0/4857 [00:00<?, ?it/s]

WV3 validation:   0%|          | 0/540 [00:00<?, ?it/s]

Epoch 001 | Train 0.020354 | PSNR 32.616 | SSIM 0.90262 | SAM 4.932° | ERGAS 4.049 | L1 0.01593
Saved best PSNR checkpoint.
Saved best balanced checkpoint.
Long training completed.


## التقييم النهائي لأفضل Checkpoint

In [33]:

best_checkpoint = torch.load(
    BEST_PSNR_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    best_checkpoint["ema_state_dict"],
    strict=True,
)

final_metrics = evaluate(model)

payload = {
    "checkpoint_epoch": int(best_checkpoint["epoch"]),
    "selection": "best EMA validation PSNR",
    "metrics": final_metrics,
    "training_profile": RUN_PROFILE,
    "target_total_epochs": TARGET_TOTAL_EPOCHS,
}

with open(FINAL_METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(
        payload,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(json.dumps(payload, indent=2, ensure_ascii=False))
print("Best PSNR checkpoint:", BEST_PSNR_PATH)
print("Best balanced checkpoint:", BEST_BALANCED_PATH)
print("Final metrics:", FINAL_METRICS_PATH)


WV3 validation:   0%|          | 0/540 [00:00<?, ?it/s]

{
  "checkpoint_epoch": 1,
  "selection": "best EMA validation PSNR",
  "metrics": {
    "l1": 0.015932725644153024,
    "psnr": 32.61632364944175,
    "ssim": 0.9026221693113998,
    "sam": 4.931671797401375,
    "ergas": 4.048586983482043
  },
  "training_profile": "smoke",
  "target_total_epochs": 1
}
Best PSNR checkpoint: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_HAT_Long_Pretraining/best_psnr_wv3_hat_long.pth
Best balanced checkpoint: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_HAT_Long_Pretraining/best_balanced_wv3_hat_long.pth
Final metrics: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_HAT_Long_Pretraining/final_validation_metrics.json



# طريقة التشغيل

## أول اختبار

```python
RUN_PROFILE = "smoke"
```

المفترض يكمل من النموذج القديم إلى Epoch 7 فقط.

## التدريب القوي

بعد نجاح الاختبار:

```python
RUN_PROFILE = "long"
```

للوصول إلى 150 Epoch.

## أقصى تشغيل

```python
RUN_PROFILE = "max"
```

للوصول إلى 300 Epoch.

يمكنك إيقاف Colab وتشغيلها لاحقًا؛ ستكمل من:

```text
WV3_HAT_Long_Pretraining/last_wv3_hat_long.pth
```

بعد انتهاء التدريب استخدم Notebook 16B لنقل أفضل EMA Checkpoint إلى البيانات المحلية.
